In [ ]:
import requests
import time
import os
import pickle
from typing import Dict, List, Any, Iterable, Optional, Tuple
from collections import defaultdict

In [ ]:
OPENALEX_BASE = "https://api.openalex.org"
MAILTO = "you@yourLaboratory.org"

In [ ]:
def get_json_with_retry(
    endpoint: str,
    params: dict,
    *,
    max_tries: int = 8,
    backoff_s: float = 0.5,
    timeout_s: float = 30.0,
) -> dict:
    """
    Basic OpenAlex GET with retry/backoff on transient errors and 429 rate limits.
    """
    url = f"{OPENALEX_BASE}/{endpoint}"
    last_err = None
    for k in range(max_tries):
        try:
            r = requests.get(url, params=params, timeout=timeout_s)
            if r.status_code == 429:
                # Respect Retry-After if provided; otherwise exponential backoff
                ra = r.headers.get("Retry-After")
                sleep_s = float(ra) if ra is not None else backoff_s * (2**k)
                time.sleep(sleep_s)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(backoff_s * (2**k))
    raise RuntimeError(f"OpenAlex request failed after {max_tries} tries: {last_err}") from last_err

In [ ]:
def _norm_source_id(source_id: str) -> str:
    """Accept 'S..' or 'https://openalex.org/S..' and return 'S..' (uppercase)."""
    sid = source_id.split("/")[-1].strip()
    if sid and sid[0].lower() == "s":
        sid = "S" + sid[1:]
    return sid


def _fetch_ids(filter_str: str, mailto: str, per_page: int, sleep_s: float) -> List[str]:
    work_ids = []
    cursor = "*"
    while cursor:
        params = {
            "filter": filter_str,
            "select": "id",
            "per_page": per_page,
            "cursor": cursor,
            "mailto": mailto,
        }
        data = get_json_with_retry("works", params)
        for w in data.get("results", []):
            work_ids.append(w["id"].split("/")[-1])
        cursor = data.get("meta", {}).get("next_cursor")
        if sleep_s:
            time.sleep(sleep_s)

    # de-dup preserving order
    seen, uniq = set(), []
    for wid in work_ids:
        if wid not in seen:
            seen.add(wid)
            uniq.append(wid)
    return uniq


def get_journal_work_ids_in_year(
    source_id: str,
    year: int,
    mailto: str,
    per_page: int = 200,
    sleep_s: float = 0.05,
) -> List[str]:
    """
    Work IDs (W...) for journal articles in a given journal (source) and year.

    Tries primary_location.source.id first (version-of-record), then falls back to
    locations.source.id if needed.
    """
    sid = _norm_source_id(source_id)

    f_primary = f"primary_location.source.id:{sid},publication_year:{year}"
    ids = _fetch_ids(f_primary, mailto, per_page, sleep_s)
    if ids:
        return ids

    f_anyloc = f"locations.source.id:{sid},publication_year:{year},type:journal-article"
    return _fetch_ids(f_anyloc, mailto, per_page, sleep_s)


In [ ]:
def get_country_work_ids_in_year(
    country_code: str,
    year: int,
    mailto: str,
    *,
    max_works: int = 6_000,
    per_page: int = 200,
    sleep_s: float = 0.05,
) -> List[str]:
    """
    Return up to max_works OpenAlex Work IDs (W...) for works that have at least one
    affiliated institution in `country_code` and publication_year == year.

    country_code: ISO-3166 alpha-2 (e.g., "ES", "FR", "US") - case-insensitive.
    """
    cc = country_code.strip().upper()
    if not cc or len(cc) != 2:
        raise ValueError("country_code must be ISO-3166 alpha-2, e.g. 'ES'.")

    if max_works <= 0:
        return []

    work_ids: List[str] = []
    cursor = "*"

    # IMPORTANT: per_page max is 200 in OpenAlex
    per_page = max(1, min(int(per_page), 200))

    while cursor and len(work_ids) < max_works:
        params = {
            "filter": f"authorships.institutions.country_code:{cc},publication_year:{int(year)}",
            "select": "id",
            "per_page": per_page,
            "cursor": cursor,
            "mailto": mailto,
        }

        data = get_json_with_retry("works", params)

        results = data.get("results", [])
        for w in results:
            work_ids.append(w["id"].split("/")[-1])  # W...
            if len(work_ids) >= max_works:
                break

        cursor = data.get("meta", {}).get("next_cursor")

        # Extra safety: if the API returns no results but still provides a cursor, break to avoid loops
        if not results:
            break

        if sleep_s:
            time.sleep(sleep_s)

    # De-duplicate while preserving order (safe)
    seen, uniq = set(), []
    for wid in work_ids:
        if wid not in seen:
            seen.add(wid)
            uniq.append(wid)

    # Respect max_works after de-dup
    return uniq[:max_works]

In [ ]:
def _chunks(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def _short_openalex_id(url_or_id: str) -> str:
    return url_or_id.split("/")[-1]

def get_institution_ids_in_city_search(
    city: str,
    country_code: str,
    mailto: str,
    *,
    per_page: int = 200,
    sleep_s: float = 0.05,
    max_pages: int = 50,
) -> List[str]:
    """
    Institutions in a city via search + post-filter on geo.city
    (because geo.city is not filterable on /institutions).
    """
    city_norm = city.strip()
    cc = country_code.strip().lower()
    if not city_norm:
        raise ValueError("city must be non-empty")
    if len(cc) != 2:
        raise ValueError("country_code must be ISO-2, e.g. 'ES'")

    per_page = max(1, min(int(per_page), 200))

    ids: List[str] = []
    cursor = "*"
    pages = 0

    while cursor and pages < max_pages:
        params = {
            "search": city_norm,
            "filter": f"country_code:{cc}",
            "select": "id,geo",
            "per_page": per_page,
            "cursor": cursor,
            "mailto": mailto,
        }
        data = get_json_with_retry("institutions", params)
        results = data.get("results", [])

        for inst in results:
            geo = inst.get("geo") or {}
            if (geo.get("city") or "").strip().lower() == city_norm.lower():
                ids.append(_short_openalex_id(inst["id"]))  # I...

        cursor = data.get("meta", {}).get("next_cursor")
        pages += 1
        if not results:
            break
        if sleep_s:
            time.sleep(sleep_s)

    # de-dup preserving order
    seen, uniq = set(), []
    for iid in ids:
        if iid not in seen:
            seen.add(iid)
            uniq.append(iid)
    return uniq

def get_work_ids_for_institution_in_year(
    inst_id: str,
    year: int,
    mailto: str,
    *,
    max_per_inst: int = 1000,
    per_page: int = 200,
    sleep_s: float = 0.05,
) -> List[str]:
    """
    Up to max_per_inst works for a single institution in a given year.
    """
    iid = _short_openalex_id(inst_id)  # I...
    per_page = max(1, min(int(per_page), 200))
    if max_per_inst <= 0:
        return []

    work_ids: List[str] = []
    cursor = "*"

    while cursor and len(work_ids) < max_per_inst:
        params = {
            "filter": f"institutions.id:{iid},publication_year:{int(year)}",
            "select": "id",
            "per_page": min(per_page, max_per_inst - len(work_ids)),
            "cursor": cursor,
            "mailto": mailto,
        }
        data = get_json_with_retry("works", params)
        results = data.get("results", [])

        for w in results:
            work_ids.append(_short_openalex_id(w["id"]))  # W...
            if len(work_ids) >= max_per_inst:
                break

        cursor = data.get("meta", {}).get("next_cursor")

        # safety: break if server returns empty page
        if not results:
            break

        if sleep_s:
            time.sleep(sleep_s)

    # de-dup preserving order
    seen, uniq = set(), []
    for wid in work_ids:
        if wid not in seen:
            seen.add(wid)
            uniq.append(wid)
    return uniq

def get_city_work_ids_in_year_capped_per_institution(
    city: str,
    country_code: str,
    year: int,
    mailto: str,
    *,
    max_per_inst: int = 1000,
    max_total: Optional[int] = None,  # optional safety cap across all institutions
    per_page: int = 200,
    sleep_s: float = 0.05,
    max_inst_pages: int = 50,
) -> Dict[str, List[str]]:
    """
    Returns a dict: inst_id -> list of W... (up to max_per_inst each)
    for institutions in city+country and works in the given year.

    Also returns only unique works per institution list; across institutions
    there may be overlap (collaborations). If you want a global unique set,
    you can union them.
    """
    inst_ids = get_institution_ids_in_city_search(
        city, country_code, mailto,
        per_page=per_page, sleep_s=sleep_s, max_pages=max_inst_pages
    )

    inst_to_works: Dict[str, List[str]] = {}
    total_seen = 0

    for iid in inst_ids:
        if max_total is not None and total_seen >= max_total:
            break

        wids = get_work_ids_for_institution_in_year(
            iid, year, mailto,
            max_per_inst=max_per_inst,
            per_page=per_page,
            sleep_s=sleep_s,
        )

        # optional trim to respect max_total (global)
        if max_total is not None and total_seen + len(wids) > max_total:
            wids = wids[: max_total - total_seen]

        inst_to_works[iid] = wids
        total_seen += len(wids)

    return inst_to_works

  def get_works_city

In [ ]:
listPublications = []
y0 = 2021
yF = 2026
option = "Journals"
if option == "Journals":
  for year in range(y0, yF):
    print(year)
    listPublications+=get_journal_work_ids_in_year("s3880285", year, MAILTO) # Science
    listPublications+=get_journal_work_ids_in_year("s137773608", year, MAILTO) # Nature
elif option == "Country":
  for year in range(y0, yF):
    print(year)
    listPublications +=get_country_work_ids_in_year("ES", year, MAILTO)
elif option == "Region":
  city = "Barcelona"
  country_code = "ES"
  years = range(y0, yF)
  max_per_inst = 1000           # cap per institution per year

  # 1) Get city institutions ONCE
  inst_ids = get_institution_ids_in_city_search(
      city=city,
      country_code=country_code,
      mailto=MAILTO,
      max_pages=100,     # adjust if needed
  )

  print(f"Institutions in {city}, {country_code}: {len(inst_ids)}")

  # 2) For each year, get works for all institutions with a cap per institution
  inst_to_works_by_year = {}  # (year -> {I...: [W...]})
  listPublications = []
  seen = set()

  for year in years:
      print(f"\nYear {year}")

      inst_to_works = {}
      for iid in inst_ids:
          wids = get_work_ids_for_institution_in_year(
              inst_id=iid,
              year=year,
              mailto=MAILTO,
              max_per_inst=max_per_inst,
          )
          inst_to_works[iid] = wids

          # accumulate globally (dedup)
          for wid in wids:
              if wid not in seen:
                  seen.add(wid)
                  listPublications.append(wid)

      inst_to_works_by_year[year] = inst_to_works

      n_year = sum(len(v) for v in inst_to_works.values())
      print(f"  works retrieved this year (raw, incl duplicates across inst): {n_year}")
      print(f"  cumulative unique works (across years & institutions): {len(listPublications)}")
elif option == "Domain":
  for year in range(y0,yF):
    print(year)
    listPublications+=get_journal_work_ids_in_year("s89954039", year, MAILTO) # JSB
    listPublications +=get_journal_work_ids_in_year("s175155502", year, MAILTO) # COSB
    listPublications +=get_journal_work_ids_in_year("s139253143", year, MAILTO) # NSMB
    listPublications +=get_journal_work_ids_in_year("s4393918103", year, MAILTO) # Acta D
    listPublications +=get_journal_work_ids_in_year("s4210171526", year, MAILTO) # Acta F

In [ ]:
print(len(listPublications))

In [ ]:
WID = str
TID = str
FID = str
SFID = str
TopicScore = Tuple[TID, float]

def _short_openalex_id(url_or_id: str) -> str:
    # "https://openalex.org/T10044" -> "T10044"
    return url_or_id.split("/")[-1]

def _load_checkpoint(pkl_path: str):
    if not os.path.exists(pkl_path):
        return None
    with open(pkl_path, "rb") as f:
        return pickle.load(f)

def _atomic_save(obj, pkl_path: str):
    tmp = pkl_path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, pkl_path)  # atomic on POSIX


def update_work_taxonomy_dicts(
    work_ids: List[str],
    mailto: str,
    primary_topics: Dict[WID, List[TopicScore]],
    all_topics: Dict[WID, List[TopicScore]],
    fields: Dict[WID, List[FID]],
    subfields: Dict[WID, List[SFID]],
    work_year: Dict[WID, int],
    topic_names: Dict[TID, str],
    field_names: Dict[FID, str],
    subfield_names: Dict[SFID, str],
    *,
    sleep_s: float = 0.05,
    save_every: int = 100,
    pkl_path: str = "dictionaries.pkl",
) -> None:
    """
    Incrementally updates dictionaries with checkpointing and resume.
    """

    # -------------------------------------------------
    # 🔹 LOAD CHECKPOINT IF EXISTS
    # -------------------------------------------------
    chk = _load_checkpoint(pkl_path)
    if chk is not None:
        print(f"[resume] Loading existing checkpoint: {pkl_path}")
        primary_topics.update(chk["primary_topics"])
        all_topics.update(chk["all_topics"])
        fields.update(chk["fields"])
        subfields.update(chk["subfields"])
        work_year.update(chk["work_year"])
        topic_names.update(chk["topic_names"])
        field_names.update(chk["field_names"])
        subfield_names.update(chk["subfield_names"])

    params = {"select": "id,publication_year,primary_topic,topics", "mailto": mailto}

    processed_since_save = 0
    total_done = len(work_year)

    # -------------------------------------------------
    # 🔹 MAIN LOOP
    # -------------------------------------------------
    for wid_in in work_ids:
        w = _short_openalex_id(wid_in)

        # Skip already processed
        if w in work_year:
            continue

        data = get_json_with_retry(f"works/{w}", params)

        # ---- year ----
        y = data.get("publication_year")
        if y is not None:
            work_year[w] = int(y)

        # ---- primary topic ----
        pt = data.get("primary_topic")
        if pt and pt.get("id"):
            tid = _short_openalex_id(pt["id"])
            score = float(pt.get("score", 0.0))
            primary_topics[w] = [(tid, score)]

            if "display_name" in pt:
                topic_names.setdefault(tid, pt["display_name"])

            sf = pt.get("subfield")
            if sf and sf.get("id"):
                sfid = _short_openalex_id(sf["id"])
                subfield_names.setdefault(sfid, sf.get("display_name", ""))

            f = pt.get("field")
            if f and f.get("id"):
                fid = _short_openalex_id(f["id"])
                field_names.setdefault(fid, f.get("display_name", ""))
        else:
            primary_topics[w] = []

        # ---- all topics ----
        topics = data.get("topics") or []
        all_list: List[TopicScore] = []
        fset, sfset = set(), set()

        for t in topics:
            if not t.get("id"):
                continue

            tid = _short_openalex_id(t["id"])
            all_list.append((tid, float(t.get("score", 0.0))))

            if "display_name" in t:
                topic_names.setdefault(tid, t["display_name"])

            sf = t.get("subfield")
            if sf and sf.get("id"):
                sfid = _short_openalex_id(sf["id"])
                sfset.add(sfid)
                subfield_names.setdefault(sfid, sf.get("display_name", ""))

            f = t.get("field")
            if f and f.get("id"):
                fid = _short_openalex_id(f["id"])
                fset.add(fid)
                field_names.setdefault(fid, f.get("display_name", ""))

        all_topics[w] = all_list
        fields[w] = list(fset)
        subfields[w] = list(sfset)

        # -------------------------------------------------
        # 🔹 CHECKPOINT SAVE
        # -------------------------------------------------
        processed_since_save += 1
        total_done += 1

        if processed_since_save >= save_every:
            print(f"[checkpoint] processed={total_done}")
            _atomic_save(
                dict(
                    primary_topics=primary_topics,
                    all_topics=all_topics,
                    fields=fields,
                    subfields=subfields,
                    work_year=work_year,
                    topic_names=topic_names,
                    field_names=field_names,
                    subfield_names=subfield_names,
                ),
                pkl_path,
            )
            processed_since_save = 0

        if sleep_s:
            time.sleep(sleep_s)

    # -------------------------------------------------
    # 🔹 FINAL SAVE
    # -------------------------------------------------
    print("[final save]")
    _atomic_save(
        dict(
            primary_topics=primary_topics,
            all_topics=all_topics,
            fields=fields,
            subfields=subfields,
            work_year=work_year,
            topic_names=topic_names,
            field_names=field_names,
            subfield_names=subfield_names,
        ),
        pkl_path,
    )

In [ ]:
primary_topics = {}
all_topics = {}
fields = {}
subfields = {}
work_year = {}

topic_names = {}
field_names = {}
subfield_names = {}

update_work_taxonomy_dicts(
    work_ids=listPublications,
    mailto=MAILTO,
    primary_topics=primary_topics,
    all_topics=all_topics,
    fields=fields,
    subfields=subfields,
    work_year=work_year,
    topic_names=topic_names,
    field_names=field_names,
    subfield_names=subfield_names,
)


In [ ]:
WID = str
TID = str
FID = str
SFID = str
TopicScore = Tuple[TID, float]

def build_yearly_counts(
    all_topics: Dict[WID, List[TopicScore]],
    fields: Dict[WID, List[FID]],
    subfields: Dict[WID, List[SFID]],
    work_year: Dict[WID, int],
    *,
    densify: bool = True,
) -> tuple[Dict[TID, Dict[int, int]], Dict[FID, Dict[int, int]], Dict[SFID, Dict[int, int]]]:
    """
    Returns (topic_counts, field_counts, subfield_counts) where:
      topic_counts[T]   = {year: count, ...}
      field_counts[F]   = {year: count, ...}
      subfield_counts[S]= {year: count, ...}

    Counts are per-work presence:
      - topics: counts 1 per (work, topic) regardless of topic score
      - fields/subfields: counts 1 per (work, field/subfield)

    Years are inferred from work_year (no user input).
    If densify=True, fills missing years with 0 for every entity.
    """
    years: Set[int] = set(work_year.values())

    topic_counts = defaultdict(lambda: defaultdict(int))
    field_counts = defaultdict(lambda: defaultdict(int))
    subfield_counts = defaultdict(lambda: defaultdict(int))

    for w, y in work_year.items():
        # Topics: ensure uniqueness per work (usually already unique, but safe)
        for tid, _score in (all_topics.get(w) or []):
            topic_counts[tid][y] += 1

        # Fields/Subfields: uniqueness per work is assumed; still safe if you want set()
        for fid in (fields.get(w) or []):
            field_counts[fid][y] += 1

        for sfid in (subfields.get(w) or []):
            subfield_counts[sfid][y] += 1

    if densify and years:
        years_sorted = sorted(years)

        def _densify(d):
            for k, yd in d.items():
                for yr in years_sorted:
                    yd.setdefault(yr, 0)

        _densify(topic_counts)
        _densify(field_counts)
        _densify(subfield_counts)

    # convert nested defaultdicts -> normal dicts (nicer to serialize / print)
    topic_counts = {k: dict(v) for k, v in topic_counts.items()}
    field_counts = {k: dict(v) for k, v in field_counts.items()}
    subfield_counts = {k: dict(v) for k, v in subfield_counts.items()}

    return topic_counts, field_counts, subfield_counts

In [ ]:
topic_counts, field_counts, subfield_counts = build_yearly_counts(
    all_topics=all_topics,
    fields=fields,
    subfields=subfields,
    work_year=work_year,
    densify=True,
)



In [ ]:
import csv
import numpy as np

def _global_years_from_counts(*counts_dicts: Dict[str, Dict[int, int]]) -> List[int]:
    years = set()
    for d in counts_dicts:
        for yd in d.values():
            years.update(yd.keys())
    return sorted(years)


def _x_for_years(years: List[int]) -> np.ndarray:
    # Requested mapping for 5 years.
    if len(years) != 5:
        raise ValueError(f"Expected 5 years, got {len(years)}: {years}")
    return np.array([-2, -1, 0, 1, 2], dtype=float)


def _fit_line(xs: np.ndarray, ys: np.ndarray) -> Tuple[float, float]:
    """
    Returns (intercept, slope) for y = intercept + slope * x.
    """
    slope, intercept = np.polyfit(xs, ys.astype(float), deg=1)
    return float(intercept), float(slope)


def fit_counts_and_save_csv(
    topic_counts: Dict[str, Dict[int, int]],
    field_counts: Dict[str, Dict[int, int]],
    subfield_counts: Dict[str, Dict[int, int]],
    topic_names: Dict[str, str],
    field_names: Dict[str, str],
    subfield_names: Dict[str, str],
    csv_path: str = "fitted_trends.csv",
) -> None:
    """
    For each entity (topic/field/subfield), fit a line to counts over the years
    using x = [-2,-1,0,1,2] (years sorted), and save coefficients to CSV.

    Output columns:
      entity_type, entity_id, name, intercept, slope, y0, y1, y2, y3, y4, year0..year4
    """
    years = _global_years_from_counts(topic_counts, field_counts, subfield_counts)
    xs = _x_for_years(years)

    def iter_rows(entity_type: str, counts: Dict[str, Dict[int, int]], names: Dict[str, str]):
        for eid, yd in counts.items():
            ys = np.array([yd.get(y, 0) for y in years], dtype=float)
            intercept, slope = _fit_line(xs, ys)
            name = names.get(eid, "")
            yield {
                "entity_type": entity_type,
                "entity_id": eid,
                "name": name,
                "intercept": intercept,
                "slope": slope,
                **{f"count_{y}": int(yd.get(y, 0)) for y in years},
            }

    rows = list(iter_rows("topic", topic_counts, topic_names))
    rows += list(iter_rows("field", field_counts, field_names))
    rows += list(iter_rows("subfield", subfield_counts, subfield_names))

    # Write CSV
    fieldnames = (
        ["entity_type", "entity_id", "name", "intercept", "slope"]
        + [f"count_{y}" for y in years]
    )

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


In [ ]:
fit_counts_and_save_csv(
    topic_counts=topic_counts,
    field_counts=field_counts,
    subfield_counts=subfield_counts,
    topic_names=topic_names,
    field_names=field_names,
    subfield_names=subfield_names,
    csv_path="fitted_trends.csv",
)


In [ ]:
from collections import Counter

WID = str
TID = str
FID = str
SFID = str
TopicScore = Tuple[TID, float]

def save_overall_label_fractions_csv(
    all_topics: Dict[WID, List[TopicScore]],
    fields: Dict[WID, List[FID]],
    subfields: Dict[WID, List[SFID]],
    work_year: Dict[WID, int],
    topic_names: Dict[TID, str],
    field_names: Dict[FID, str],
    subfield_names: Dict[SFID, str],
    csv_path: str = "label_fractions.csv",
) -> None:
    """
    Computes fraction of papers (over the full 5-year window) containing each
    topic/field/subfield and saves to CSV.

    Fraction definition:
        fraction = (# works containing label) / (total works)
    """

    works = list(work_year.keys())
    N = len(works)
    if N == 0:
        raise ValueError("No works available.")

    topic_ct = Counter()
    field_ct = Counter()
    subfield_ct = Counter()

    # -------------------------------------------------
    # Count presence per work
    # -------------------------------------------------
    for w in works:
        # topics
        for tid, _ in (all_topics.get(w) or []):
            topic_ct[tid] += 1

        # fields/subfields (unique per work)
        for fid in set(fields.get(w) or []):
            field_ct[fid] += 1

        for sfid in set(subfields.get(w) or []):
            subfield_ct[sfid] += 1

    # -------------------------------------------------
    # Prepare rows
    # -------------------------------------------------
    rows = []

    def add_rows(entity_type, counter, names):
        for eid, cnt in counter.items():
            rows.append({
                "entity_type": entity_type,
                "entity_id": eid,
                "name": names.get(eid, ""),
                "fraction": cnt / N,
                "count": cnt,
                "total_works": N,
            })

    add_rows("topic", topic_ct, topic_names)
    add_rows("field", field_ct, field_names)
    add_rows("subfield", subfield_ct, subfield_names)

    # sort by fraction descending (nice for inspection)
    rows.sort(key=lambda r: r["fraction"], reverse=True)

    # -------------------------------------------------
    # Write CSV
    # -------------------------------------------------
    fieldnames = [
        "entity_type",
        "entity_id",
        "name",
        "fraction",
        "count",
        "total_works",
    ]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"[saved] {csv_path} ({len(rows)} rows)")

In [ ]:
save_overall_label_fractions_csv(
    all_topics=all_topics,
    fields=fields,
    subfields=subfields,
    work_year=work_year,
    topic_names=topic_names,
    field_names=field_names,
    subfield_names=subfield_names,
    csv_path="label_fractions.csv",
)